# 05 Scoring Thresholds and Quality Flags

Fase 5 y 6: umbrales interpretativos y validaciones QA.

In [1]:
from pathlib import Path
import polars as pl
root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd()
scores = pl.read_parquet(root / 'outputs' / 'state_scores_long.parquet')
scores.head()

state_code,state_name,pillar,criterion_id,criterion_name,survey_used,fallback_used,estimate_pct,score_100,threshold_band,optional_score_0_1_2,low_reliability_flag,reliability_reason,unweighted_n,weighted_population_if_available,se_if_available,ci_low_if_available,ci_high_if_available,notes
str,str,str,str,str,str,bool,f64,f64,str,i32,bool,str,i64,f64,f64,f64,f64,str
"""20""","""Oaxaca""","""Pillar 4""","""M1""","""Depressive / anxiety / stress …","""ENDISEG_2021""",false,65.011097,34.988903,"""high""",2,false,"""""",1381,3.063477e6,1.475697,62.118731,67.903463,"""Tertiles: <= 27.95 low, <= 32.…"
"""08""","""Chihuahua""","""Pillar 4""","""M1""","""Depressive / anxiety / stress …","""ENDISEG_2021""",false,73.392408,26.607592,"""low""",0,false,"""""",1357,2.856632e6,1.346784,70.752711,76.032104,"""Tertiles: <= 27.95 low, <= 32.…"
"""12""","""Guerrero""","""Pillar 4""","""M1""","""Depressive / anxiety / stress …","""ENDISEG_2021""",false,66.127639,33.872361,"""high""",2,false,"""""",1292,2.512364e6,1.524373,63.139869,69.11541,"""Tertiles: <= 27.95 low, <= 32.…"
"""30""","""Veracruz de Ignacio de la Llav…","""Pillar 4""","""M1""","""Depressive / anxiety / stress …","""ENDISEG_2021""",false,65.929204,34.070796,"""high""",2,false,"""""",1168,6.393998e6,1.560705,62.870222,68.988186,"""Tertiles: <= 27.95 low, <= 32.…"
"""24""","""San Luis Potosí""","""Pillar 4""","""M1""","""Depressive / anxiety / stress …","""ENDISEG_2021""",false,59.02205,40.97795,"""high""",2,false,"""""",1447,2.149517e6,1.473529,56.133933,61.910167,"""Tertiles: <= 27.95 low, <= 32.…"


In [2]:
# QA checks
qa = {
    'rows': scores.height,
    'states_detected': scores.select(pl.col('state_code').n_unique()).item(),
    'criteria_detected': scores.select(pl.col('criterion_id').n_unique()).item(),
    'impossible_pct': scores.filter((pl.col('estimate_pct') < 0) | (pl.col('estimate_pct') > 100)).height,
}
qa

{'rows': 192,
 'states_detected': 32,
 'criteria_detected': 6,
 'impossible_pct': 0}

In [3]:
scores.group_by(['criterion_id', 'threshold_band']).len().sort(['criterion_id', 'threshold_band'])

criterion_id,threshold_band,len
str,str,u32
"""M1""","""high""",10
"""M1""","""low""",11
"""M1""","""medium""",11
"""M2""","""high""",10
"""M2""","""low""",11
…,…,…
"""S2""","""low""",11
"""S2""","""medium""",11
"""S3""","""high""",10


In [4]:
scores.group_by('criterion_id').agg([
    pl.mean('low_reliability_flag').alias('share_low_reliability'),
    pl.mean('unweighted_n').alias('mean_unweighted_n')
]).sort('criterion_id')

criterion_id,share_low_reliability,mean_unweighted_n
str,f64,f64
"""M1""",0.0,1380.90625
"""M2""",0.0,1380.90625
"""M3""",0.71875,964.84375
"""S1""",0.0,1129.78125
"""S2""",0.0,1129.78125
"""S3""",0.96875,1380.90625
